# FORGE — Report di Strategia (single-event)

Dato **il contratto di un evento**, questo notebook esegue **Alpha Discovery** (Modulo 2) e
**Rule Discovery** (Modulo 3) e produce un report di sintesi con tutte le metriche rilevanti:

1. Configurazione e caricamento dati (+ Market Context / regimi)
2. Costruzione del contratto-evento
3. **Alpha Discovery** — target derivato, grade A–D, IC, edge statistico, OOS, profilo per orizzonte
4. **Rule Discovery** — verdetto economico (EDGE / PARTIAL-EDGE / NON-EDGE)
5. **Andamento nel tempo** — equity, drawdown, P&L mensile, rolling win-rate/PF
6. **Distribuzione dei trade** ed **escursioni** (MAE/MFE)
7. **Comportamento nei regimi** di mercato
8. **Walk-forward** out-of-sample e **validazione statistica**
9. **Landscape della griglia** e report testuale completo

> Per analizzare un'altra strategia basta modificare la cella *Configurazione*: il path dei dati e
> la funzione `build_event_series` che definisce l'evento.

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from forgedge import (MarketContext, AlphaDiscovery, AlphaConfig, RuleDiscovery,
                      RuleDiscoveryConfig, SelectionCriteria)
from forgedge.event_discovery.models import (
    EventCandidate, EventComponent, ActivationStats, GateResult)
from forgedge.rule_discovery.backtest import run_backtest
from forgedge.rule_discovery.grid import select_best
from forgedge.rule_discovery.report import text_report, html_report

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True,
                     "grid.alpha": 0.3, "figure.dpi": 110})

def metric_table(d, title=None):
    """Render a {label: value} dict as a one-column DataFrame."""
    df = pd.DataFrame({"value": pd.Series(d, dtype="object")})
    if title: display(Markdown(f"**{title}**"))
    display(df)
    return df

def pf(net):
    net = np.asarray(net, float)
    pos, neg = net[net > 0].sum(), -net[net < 0].sum()
    return (pos / neg) if neg > 0 else (np.inf if pos > 0 else np.nan)

## 1. Configurazione

Definisci qui il dataset, l'asset e — soprattutto — **il contratto dell'evento** come serie
booleana sul DataFrame KPI. L'esempio di default è un setup *oversold* su ADAUSDC 1h:
`close_rsi_14 < 34.17  AND  pctrank(close_rsi_25, 96) < 0.104`.

In [ ]:
DATA_PATH      = "/root/.claude/uploads/d4188b96-ea1b-59e8-a812-6ea04c87ddc3/8c25da9f-test1h.xlsx"
SYMBOL         = "ADAUSDC"        # set None to keep all rows
TIMEFRAME      = "1H"
TIMESTAMP_COL  = "open_time"      # raw timestamp column
TIMESTAMP_UNIT = "ms"             # "ms" for epoch-millis, None if already datetime

# ---- THE EVENT CONTRACT ------------------------------------------------------
EVENT_EXPRESSION     = "close_rsi_14 < 34.1718 AND pctrank(close_rsi_25, 96) < 0.104167"
EVENT_SOURCE_FEATURE = "close_rsi_14"

def build_event_series(df: pd.DataFrame) -> pd.Series:
    """Return a 0/1/NaN boolean activation series aligned to df.index."""
    rsi14 = df["close_rsi_14"].astype(float)
    pr96  = df["close_rsi_25"].astype(float).rolling(96, min_periods=48).rank(pct=True)
    ev = ((rsi14 < 34.1718) & (pr96 < 0.104167)).astype(float)
    ev[pr96.isna()] = np.nan
    return ev

## 2. Dati + Market Context (regimi)

In [ ]:
raw = pd.read_excel(DATA_PATH)
df  = raw[raw["symbol"] == SYMBOL].copy() if (SYMBOL and "symbol" in raw.columns) else raw.copy()

if TIMESTAMP_UNIT:
    df["open_dt"] = pd.to_datetime(df[TIMESTAMP_COL], unit=TIMESTAMP_UNIT)
else:
    df["open_dt"] = pd.to_datetime(df[TIMESTAMP_COL])
df = df.sort_values("open_dt").set_index("open_dt")

# Market Context — append `regime` / `regime_stable`. Degrade gracefully.
try:
    df = MarketContext(df).run()
    if not isinstance(df.index, pd.DatetimeIndex):
        df = df.set_index("open_dt")
    regime_ok = "regime" in df.columns
except Exception as e:
    print("Market Context skipped:", e)
    df["regime"] = "ALL"; regime_ok = False

print(f"{SYMBOL}: {len(df):,} bars   {df.index[0]}  ->  {df.index[-1]}")
if regime_ok:
    display(df["regime"].value_counts(normalize=True).mul(100).round(1).to_frame("% bars"))

## 3. Contratto-evento

In [ ]:
ev    = build_event_series(df)
n_act = int(np.nansum(ev.values))
EVENT_ID = f"EVT-{SYMBOL}-report"

comp = EventComponent(
    source_feature=EVENT_SOURCE_FEATURE, transform="identity", transform_params={},
    transformed_col=EVENT_SOURCE_FEATURE, threshold=float("nan"), threshold_type="custom",
    direction="below", event_type="threshold", expression=EVENT_EXPRESSION)
cand = EventCandidate(
    EVENT_ID, "CANDIDATE", [comp], EVENT_EXPRESSION,
    ActivationStats(n_act, 12, 0, 0.2, 5.0), GateResult(True, n_act, 12, 0.2, 5.0),
    event_series=ev)

display(Markdown(f"**Event:** `{EVENT_EXPRESSION}`"))
print(f"activations: {n_act:,}  ({n_act/len(df):.2%} of bars)")

## 4. Alpha Discovery — edge statistico e target derivato

In [ ]:
ad = AlphaDiscovery(df.copy(), [cand], AlphaConfig(asset=SYMBOL, timeframe=TIMEFRAME))
contract = ad.run()[0]
dt, ic, es = contract.derived_target, contract.underlying_feature, contract.event_stats

metric_table({
    "grade (A-D)":        contract.alpha_score.grade,
    "composite score":    round(contract.alpha_score.composite_score, 4),
    "promoted":           contract.promoted,
    "direction":          dt.direction,
    "holding h*":         dt.holding_period_h,
    "sell_pct (TP)":      f"{dt.sell_pct*100:.2f}%",
    "mean advantage":     f"{dt.mean_advantage*100:.3f}%",
    "base rate":          f"{contract.base_rate:.3f}",
}, "Target derivato")

metric_table({
    "IC":                 round(ic.ic, 4),
    "IC p-value":         round(ic.p_value, 4),
    "IC admitted":        ic.admitted,
    "rolling IC stable":  ic.rolling_ic_stable,
    "sign consistency":   round(ic.rolling_sign_consistency, 3),
    "lift":               round(es.lift, 4),
    "cohen's d":          round(es.cohens_d, 4),
    "win / base rate":    f"{es.win_rate:.3f} / {es.base_rate:.3f}",
    "t-stat / p-value":   f"{es.t_stat:.3f} / {es.p_value:.4f}",
}, "Misure di edge (in-sample)")

if contract.oos_validation is not None:
    o = contract.oos_validation
    metric_table({
        "OOS passed":     o.passed,
        "OOS mean adv":   f"{o.mean_advantage*100:.3f}%",
        "OOS lift":       round(o.lift, 4),
        "OOS win/base":   f"{o.win_rate:.3f} / {o.base_rate:.3f}",
        "OOS t / p":      f"{o.t_stat:.3f} / {o.p_value:.4f}",
    }, "Conferma out-of-sample (coda 30%)")

if contract.rejection_reasons:
    display(Markdown("**Diagnostica (non bloccante):** " +
                     "; ".join(contract.rejection_reasons)))

### 4.1 Selezione dell'orizzonte  `h* = argmax(|adv| / √h)`

In [ ]:
hs   = list(dt.advantage_by_h)
adv  = np.array([dt.advantage_by_h[h] for h in hs]) * 100
sc   = np.array([dt.score_by_h[h]    for h in hs])
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
c = ["#d62728" if h == dt.holding_period_h else "#7f9bbf" for h in hs]
ax[0].bar(range(len(hs)), sc, color=c); ax[0].set_xticks(range(len(hs))); ax[0].set_xticklabels(hs)
ax[0].set_title("score |adv|/√h  (rosso = h* scelto)"); ax[0].set_xlabel("horizon (bars)")
ax[1].bar(range(len(hs)), adv, color=c); ax[1].set_xticks(range(len(hs))); ax[1].set_xticklabels(hs)
ax[1].axhline(0, color="k", lw=0.6); ax[1].set_title("vantaggio medio punto-a-punto (%)")
ax[1].set_xlabel("horizon (bars)"); plt.tight_layout(); plt.show()

### 4.2 Edge per regime (Alpha Discovery)

In [ ]:
ra = contract.regime_analysis
if ra is not None and ra.per_regime:
    rdf = pd.DataFrame([{
        "regime": s.regime, "n": s.n, "IC": round(s.ic, 4),
        "p": round(s.p_value, 4), "win_rate": round(s.win_rate, 3),
        "strength": s.strength} for s in ra.per_regime]).set_index("regime")
    display(rdf)
    print(f"dependency_type={ra.dependency_type}  regime_breadth={ra.regime_breadth}  "
          f"weak={ra.weak_regimes}")
else:
    print("No regime breakdown available.")

## 5. Rule Discovery — verdetto economico

Eseguito con `early_elimination=False`: anche un verdetto **NON-EDGE** percorre l'intera pipeline,
così walk-forward, validazione statistica, envelope, escursioni e regimi sono **sempre** popolati e
il report è uniforme per qualsiasi strategia. Il verdetto resta solo un'etichetta.

In [ ]:
rd   = RuleDiscovery(df.copy(), contract, cand,
                     config=RuleDiscoveryConfig(criteria=SelectionCriteria(early_elimination=False)))
resp = rd.run()

colour = {"EDGE": "🟢", "PARTIAL-EDGE": "🟡", "NON-EDGE": "🔴"}.get(resp.verdict, "")
display(Markdown(f"## {colour} Verdetto: **{resp.verdict}**"))
if resp.rejection_reasons:
    display(Markdown("**Motivi:** " + "; ".join(resp.rejection_reasons)))
if resp.notes:
    display(Markdown("*" + " · ".join(resp.notes) + "*"))

### 5.1 Configurazione selezionata + ledger su tutto il periodo

Per le analisi temporali ricostruiamo il **ledger per-trade** della configurazione scelta
(la `validated_rule` se EDGE/PARTIAL, altrimenti la migliore combinazione di griglia). Walk-forward,
validazione, envelope, escursioni e regimi arrivano direttamente da `resp` (sempre popolati grazie a
`early_elimination=False`).

In [ ]:
def selected_params(resp, rd):
    if resp.validated_rule is not None:
        return resp.validated_rule.params
    best = select_best(resp.grid_results, rd.config.criteria)
    return best.params if best is not None else rd._seed_base_params([])

cfg    = rd.config
params = selected_params(resp, rd)

# Full-period in-sample ledger for the selected configuration (for the time-series charts).
summary, trades = run_backtest(
    rd._frame, cfg.signal_col, params,
    scoring=cfg.scoring, timestamp_col=cfg.timestamp_col, return_trades=True)
trades["fill_dt"] = pd.to_datetime(trades["fill_dt"])
trades["exit_dt"] = pd.to_datetime(trades["exit_dt"])
trades = trades.sort_values("exit_dt").reset_index(drop=True)

# Diagnostics straight from the response (populated for every verdict).
wf        = resp.walk_forward
stat_val  = resp.statistical_validation
envelope  = resp.execution_envelope
excursion = resp.excursion
regime    = resp.regime_analysis

metric_table({
    "direction":   params.direction,
    "entry":       f"{params.buy_type}  drop={params.buy_drop_pct}  delay={params.buy_delay_bar}",
    "exit":        f"sell_pct={params.sell_pct}  target_h={params.target_h}  fee={params.fee}",
    "early_stop":  params.early_stopping,
    "signals":     summary.total_signals,
    "trades (fill)": summary.total_trades,
    "fill rate":   f"{summary.fill_rate:.2%}",
    "win rate":    f"{summary.win_rate_pct if summary.win_rate_pct>1 else summary.win_rate_pct*100:.1f}%",
    "profit factor": round(summary.profit_factor, 3),
    "expectancy":  f"{summary.expectancy*100:.3f}%",
    "target hit":  f"{summary.target_hit_rate_pct:.1f}%",
    "best / worst": f"{summary.best_trade*100:.2f}% / {summary.worst_trade*100:.2f}%",
}, "Configurazione & backtest (intero periodo)")

## 6. Andamento della strategia nel tempo

In [ ]:
if len(trades):
    t   = trades.copy()
    net = t["net_pct_gain"].to_numpy()
    eq  = (1 + pd.Series(net, index=t["exit_dt"])).cumprod()
    dd  = eq / eq.cummax() - 1.0

    fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True,
                           gridspec_kw={"height_ratios": [2, 1]})
    ax[0].plot(eq.index, eq.values, color="#1f77b4")
    ax[0].axhline(1, color="k", lw=0.6)
    ax[0].set_ylabel("equity (×, compounded)")
    ax[0].set_title(f"Equity curve — {len(t)} trades — "
                    f"final {(eq.iloc[-1]-1)*100:+.1f}%  ·  maxDD {dd.min()*100:.1f}%")
    ax[1].fill_between(dd.index, dd.values*100, 0, color="#d62728", alpha=0.4)
    ax[1].set_ylabel("drawdown (%)"); ax[1].set_xlabel("exit date")
    plt.tight_layout(); plt.show()

    metric_table({
        "total return (compounded)": f"{(eq.iloc[-1]-1)*100:+.1f}%",
        "sum of trade returns":      f"{net.sum()*100:+.1f}%",
        "max drawdown":              f"{dd.min()*100:.1f}%",
        "profit factor":             round(pf(net), 3),
        "avg trade":                 f"{net.mean()*100:+.3f}%",
        "median trade":              f"{np.median(net)*100:+.3f}%",
        "volatility (per trade)":    f"{net.std()*100:.3f}%",
        "Sharpe (per-trade, raw)":   round(net.mean()/net.std(), 3) if net.std() else float("nan"),
    }, "Sintesi temporale")
else:
    print("No trades to plot.")

### 6.1 P&L mensile e rolling win-rate / profit factor

In [ ]:
if len(trades):
    t = trades.copy()
    ms = t.set_index("exit_dt")["net_pct_gain"]
    m  = ms.groupby(ms.index.to_period("M")).sum()
    m.index = m.index.to_timestamp()
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
    colours = ["#2ca02c" if v >= 0 else "#d62728" for v in m]
    ax[0].bar(m.index, m*100, width=20, color=colours)
    ax[0].axhline(0, color="k", lw=0.6); ax[0].set_title("P&L mensile (somma rendimenti %)")
    ax[0].set_ylabel("%")

    w = min(30, max(5, len(t)//4))
    roll_wr = t["net_pct_gain"].gt(0).rolling(w).mean()*100
    roll_pf = t["net_pct_gain"].rolling(w).apply(pf, raw=True)
    ax2 = ax[1]; ax3 = ax2.twinx()
    ax2.plot(t["exit_dt"], roll_wr, color="#1f77b4", label="win-rate %")
    ax3.plot(t["exit_dt"], roll_pf.clip(upper=5), color="#ff7f0e", label="PF (clip 5)")
    ax2.axhline(50, color="#1f77b4", ls=":", lw=0.8); ax3.axhline(1, color="#ff7f0e", ls=":", lw=0.8)
    ax2.set_ylabel("win-rate %", color="#1f77b4"); ax3.set_ylabel("profit factor", color="#ff7f0e")
    ax2.set_title(f"Rolling (finestra {w} trade)")
    plt.tight_layout(); plt.show()

## 7. Distribuzione dei trade & escursioni (MAE / MFE)

In [ ]:
if len(trades):
    net = trades["net_pct_gain"].to_numpy()*100
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
    ax[0].hist(net, bins=40, color="#7f9bbf", edgecolor="white")
    ax[0].axvline(0, color="k", lw=0.8)
    ax[0].axvline(net.mean(), color="#d62728", lw=1.2, label=f"mean {net.mean():+.2f}%")
    ax[0].set_title("Distribuzione rendimenti per trade (%)"); ax[0].legend()

    win = trades["target_hit"]
    ax[1].hist([net[win], net[~win]], bins=30, stacked=True,
               color=["#2ca02c", "#d62728"], label=["target hit", "time-stop exit"])
    ax[1].set_title("Esiti: take-profit vs uscita a scadenza"); ax[1].legend()
    plt.tight_layout(); plt.show()

# Excursion stats (aggregate) — computed unconditionally
ex = excursion
if ex is not None:
    metric_table({
        "n trades":            ex.n_trades,
        "MAE mean / median":   f"{ex.mae_mean*100:.2f}% / {ex.mae_median*100:.2f}%",
        "MAE worst":           f"{ex.mae_worst*100:.2f}%",
        "MFE mean / median":   f"{ex.mfe_mean*100:.2f}% / {ex.mfe_median*100:.2f}%",
        "MFE best":            f"{ex.mfe_best*100:.2f}%",
        "MFE reached target":  f"{ex.mfe_reached_target_pct:.1f}%",
    }, "Escursioni intra-trade (MAE / MFE)")

## 8. Comportamento nei regimi

Ripartizione della performance per regime di mercato sulla configurazione selezionata —
calcolata sempre, anche per un NON-EDGE.

In [ ]:
if regime is not None and regime.per_regime:
    g = pd.DataFrame(regime.per_regime).set_index("regime")
    display(g)
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
    colours = ["#2ca02c" if v >= 1 else "#d62728" for v in g["profit_factor"]]
    ax[0].bar(g.index, g["profit_factor"].clip(upper=5), color=colours)
    ax[0].axhline(1, color="k", lw=0.8); ax[0].set_title("Profit factor per regime (clip 5)")
    ax[1].bar(g.index, g["expectancy"]*100, color=colours)
    ax[1].axhline(0, color="k", lw=0.8); ax[1].set_title("Expectancy per regime (%)")
    for a in ax: a.tick_params(axis="x", rotation=30)
    plt.tight_layout(); plt.show()
    print(f"dependency_score={regime.dependency_score}  zero_months={regime.zero_months}  "
          f"avoid_in={regime.avoid_in}")
else:
    print("No regime breakdown (regime column missing or no trades).")

## 9. Walk-forward out-of-sample & validazione statistica

In [ ]:
if wf is not None:
    rows = []
    for s in wf.splits:
        ts = s.test_summary
        rows.append({"split": s.split_idx, "test_from": s.test_from[:10], "test_to": s.test_to[:10],
                     "trades": ts.total_trades, "win%": round(ts.win_rate_pct if ts.win_rate_pct>1 else ts.win_rate_pct*100,1),
                     "PF": round(ts.profit_factor, 2), "net%": round(ts.total_net_gain*100, 2)})
    display(pd.DataFrame(rows).set_index("split"))
    o = wf.oos_summary
    metric_table({
        "n splits":             len(wf.splits),
        "profitable splits":    f"{wf.n_profitable_splits}/{len(wf.splits)}",
        "consistency":          f"{wf.consistency:.0%}",
        "OOS trades":           o.total_trades,
        "OOS win rate":         f"{o.win_rate_pct if o.win_rate_pct>1 else o.win_rate_pct*100:.1f}%",
        "OOS profit factor":    round(o.profit_factor, 3),
        "OOS expectancy":       f"{o.expectancy*100:.3f}%",
    }, "Track record out-of-sample (concatenazione test windows)")
else:
    print("Walk-forward non disponibile — span dati troppo corto per uno split.")

sv = stat_val
if sv is not None:
    metric_table({
        "win-rate t / p":      f"{sv.ttest_winrate_t:.2f} / {sv.ttest_winrate_p:.4f}",
        "expectancy t / p":    f"{sv.ttest_expectancy_t:.2f} / {sv.ttest_expectancy_p:.4f}",
        "Sharpe ratio":        round(sv.sharpe_ratio, 3),
        "deflated Sharpe":     round(sv.deflated_sharpe, 3),
        "trials tested":       sv.n_trials_tested,
        "temporal stability":  sv.temporal_stability,
        "PF 1st / 2nd half":   f"{sv.pf_first_half:.2f} / {sv.pf_second_half:.2f}",
    }, "Validazione statistica")

### 9.1 Execution envelope (conservativo ↔ ottimistico)

In [ ]:
en = envelope
if en is not None:
    cmp = pd.DataFrame({
        "conservative (close)": {"trades": en.conservative.total_trades,
            "win%": round(en.conservative.win_rate_pct if en.conservative.win_rate_pct>1 else en.conservative.win_rate_pct*100,1),
            "PF": round(en.conservative.profit_factor,3),
            "net%": round(en.conservative.total_net_gain*100,2)},
        "optimistic (high)": {"trades": en.optimistic.total_trades,
            "win%": round(en.optimistic.win_rate_pct if en.optimistic.win_rate_pct>1 else en.optimistic.win_rate_pct*100,1),
            "PF": round(en.optimistic.profit_factor,3),
            "net%": round(en.optimistic.total_net_gain*100,2)}}).T
    display(cmp)
    print("La performance reale della regola sta fra i due estremi.")

## 10. Landscape della griglia

In [ ]:
if resp.grid_results:
    grid = pd.DataFrame([g.row() for g in resp.grid_results])
    display(grid.sort_values("pf_score_tpm", ascending=False).head(12).reset_index(drop=True))
    piv = grid.pivot_table(index="sell_pct", columns="target_h",
                           values="profit_factor", aggfunc="max")
    fig, ax = plt.subplots(figsize=(7, 4))
    im = ax.imshow(piv.values, cmap="RdYlGn", vmin=0.5, vmax=1.5, aspect="auto", origin="lower")
    ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels(piv.columns)
    ax.set_yticks(range(len(piv.index))); ax.set_yticklabels([f"{v:.3f}" for v in piv.index])
    ax.set_xlabel("target_h"); ax.set_ylabel("sell_pct"); ax.set_title("max PF sulla griglia")
    fig.colorbar(im, ax=ax, label="profit factor"); plt.tight_layout(); plt.show()

## 11. Report testuale completo

In [ ]:
print(text_report(resp))

### Esporta il report (HTML)

Salva una versione HTML autoconsistente del verdetto Rule Discovery.

In [ ]:
out = f"report_{SYMBOL}_{resp.verdict}.html"
with open(out, "w") as fh:
    fh.write(html_report(resp))
print("saved:", out)